# 5 — Experiment Results & Comparison
**Author:** Naga Sai Bharath Potla

This notebook consolidates all results from the baseline models, the proposed
hybrid model, and the ablation study into a unified comparison.

In [ ]:
import sys, os, json, glob
sys.path.insert(0, os.path.join(os.pardir, "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from config import RESULTS_DIR
sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 120

## 5.1 Load all results

In [ ]:
def load_json(path):
    with open(path) as f:
        data = json.load(f)
    return data.get("test", data)

results = {}

# baselines
lr_path = os.path.join(RESULTS_DIR, "baseline_logreg_metrics.json")
if os.path.exists(lr_path):
    results["Logistic Regression"] = load_json(lr_path)

bilstm_path = os.path.join(RESULTS_DIR, "baseline_bilstm_metrics.json")
if os.path.exists(bilstm_path):
    results["BiLSTM + Attention"] = load_json(bilstm_path)

bigru_path = os.path.join(RESULTS_DIR, "baseline_bigru_metrics.json")
if os.path.exists(bigru_path):
    results["BiGRU + Attention"] = load_json(bigru_path)

# proposed
hybrid_path = os.path.join(RESULTS_DIR, "hybrid_cnn_bigru_attn_metrics.json")
if os.path.exists(hybrid_path):
    results["Proposed (CNN+BiGRU+Attn)"] = load_json(hybrid_path)

# ablation
ablation_path = os.path.join(RESULTS_DIR, "ablation_results.json")
if os.path.exists(ablation_path):
    abl = json.load(open(ablation_path))
    for k, v in abl.items():
        results[f"Ablation: {k}"] = v

print(f"Loaded results for {len(results)} configurations")

## 5.2 Full comparison table

In [ ]:
rows = [{"Model": name, **metrics} for name, metrics in results.items()]
df = pd.DataFrame(rows).set_index("Model")

# highlight best per column
styled = df.style.highlight_max(axis=0, color="#d4edda")
styled = styled.format("{:.4f}")
styled

## 5.3 Grouped bar chart — all metrics

In [ ]:
# separate main models and ablation for cleaner charts
main_models = {k: v for k, v in results.items() if not k.startswith("Ablation")}
ablation_models = {k: v for k, v in results.items() if k.startswith("Ablation")}

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# main comparison
main_df = pd.DataFrame([{"Model": k, **v} for k, v in main_models.items()]).set_index("Model")
main_df.plot(kind="bar", ax=axes[0], ylim=(0.5, 1.0), rot=15)
axes[0].set_title("Main Model Comparison")
axes[0].legend(loc="lower right", fontsize=8)

# ablation
if ablation_models:
    abl_df = pd.DataFrame([{"Variant": k.replace('Ablation: ',''), **v} for k, v in ablation_models.items()]).set_index("Variant")
    abl_df.plot(kind="bar", ax=axes[1], ylim=(0.5, 1.0), rot=15)
    axes[1].set_title("Ablation Study")
    axes[1].legend(loc="lower right", fontsize=8)

plt.tight_layout()
plt.show()

## 5.4 Radar chart — main models

In [ ]:
from math import pi

metrics_list = ["accuracy", "precision", "recall", "f1", "auc"]
n_metrics = len(metrics_list)
angles = [i / n_metrics * 2 * pi for i in range(n_metrics)] + [0]

fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))
colors = plt.cm.Set2.colors

for idx, (name, met) in enumerate(main_models.items()):
    values = [met.get(m, 0) for m in metrics_list] + [met.get(metrics_list[0], 0)]
    ax.plot(angles, values, "o-", label=name, color=colors[idx % len(colors)])
    ax.fill(angles, values, alpha=0.1, color=colors[idx % len(colors)])

ax.set_xticks(angles[:-1])
ax.set_xticklabels([m.title() for m in metrics_list])
ax.set_ylim(0.5, 1.0)
ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1), fontsize=8)
ax.set_title("Model Comparison (Radar)", y=1.08)
plt.tight_layout()
plt.show()

## 5.5 Training curve comparison

In [ ]:
curve_files = {
    "BiLSTM":  os.path.join(RESULTS_DIR, "baseline_bilstm_metrics.json"),
    "BiGRU":   os.path.join(RESULTS_DIR, "baseline_bigru_metrics.json"),
    "Proposed": os.path.join(RESULTS_DIR, "hybrid_cnn_bigru_attn_metrics.json"),
}

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for name, path in curve_files.items():
    if not os.path.exists(path):
        continue
    data = json.load(open(path))
    hist = data.get("history", {})
    if "val_loss" in hist:
        axes[0].plot(hist["val_loss"], label=name)
    if "val_f1" in hist:
        axes[1].plot(hist["val_f1"], label=name)

axes[0].set_title("Validation Loss")
axes[0].set_xlabel("Epoch")
axes[0].legend()
axes[1].set_title("Validation F1")
axes[1].set_xlabel("Epoch")
axes[1].legend()
plt.tight_layout()
plt.show()

## 5.6 Key findings

| Finding | Detail |
| --- | --- |
| Best overall model | **Proposed CNN-BiGRU-Attention** |
| LLM embeddings | `all-MiniLM-L6-v2` captures rich semantic relationships |
| CNN contribution | Local feature extraction across sentence embeddings improves classification |
| Attention benefit | Sequential attention improves over mean pooling and provides interpretability |
| Baseline strength | Even Logistic Regression on mean-pooled LLM embeddings is competitive |

The proposed hybrid architecture achieves the best balance of accuracy, F1-score,
and AUC while maintaining interpretability through the attention mechanism.